# References, encapsulation, copy and move semantics

## References
References allow you to create aliases for variables, making it possible to pass and manipulate data without copying it. They differ from pointers in that they must always refer to a valid object. They cannot be null, and once created, they cannot be changed to point to another object. This makes them safer and more intuitive in may situations, especially when used as function parameters. 

References are also essential to the writing of copy constructors and assingment operators, which manage how one object state is copied into another. 

````C++
void increment(int& value){ // Pass by reference
    ++value; 
}
int main(){
    int x = 10;       
    int& ref = x;     // ref is a reference to x
    increment(ref);   // x is modified through ref
    std::cout << x;   // Output is 11
}
````

## Copy constructor (copy semantics). 
A copy constructor defines how a new object is created as a copy of an existing one. By default, c++ provides a member-wise of shallow copy constructor. This is usually sufficient for simple classes however, file handles or maybe network connections, then you'll need to implement a custom copy constructor to ensure a proper deep copy. This will help avoid issues such as double deletion, dangling pointers or shared multiple state, which can result where multiple objects point to the same underlying resource. 

<img src="./IMG/Copy_Constructor.png" width="400">

````c++
class Employee{
    private:
        char* name; 
    public:
        Employee(const Employee& other){
            name = new char[std::strlen(other.name)+1]; 
            std::strcpy(name, other.name);
        }
        };
````

## Assignment operators (move semantics). 
Assignment operators define how one existing object is assigned the state of another. As with the copy constructor, a default assignment is provided by the compiler, but it performs a shallow member-wise copy. If any resources must be released before the new values are assigned, the assignment operator is expected to take care of this. When it is finished with its task, the function will return a reference to the current object by returning `*this` to allow for function chaining.


````c++
class Employee{
    private:
        char* name; 
    public:
        Employee& operator=(const Employee& other){
            if(this != other){
                delete[] name; 
                name = new char[std::strlen(other.name)+1]; 
                std::strcpy(name, other.name); 
            }
            return *this;
        }
    };
````
A common pattern is to check for self-assignment, i.e assigning an object to itself. This could lead to deleting memory before it's being copied. If the object is not being assigned to itself, the operator performs a deep copy of resources from the source object to the destination object. 

## Move constructors

<img src="./IMG/Move_Constructor.png" width="400">

Move constructors allow ownership of resources to be transferred from one object to another without duplicating the underlying data. This avoids the overhead of deep copying, which can be expensive for types that manage dynamic memory, file handles, or other resources. This is useful when creating objects that may be stored in standard library containers like STD vector. 

When a container resizes itself or reorders the objects it contains, it may need to re-allocate the objects in memory. If move constructors are defined, the container can transfer ownership ot each object efficiently without performing deep copies. Move assignment operators allow efficient transfer of resources between two existing objects during assignment. 

````C++
class Employee{
private:
    std::string name; 
public: 
    Employee& operator=(Employee& other) noexcept{
        if (this != &other){
            name = std::move(other.name);
        }
        return *this;
    }
````

Unlike move constructors, which are called on uninitialized objects, move assignment operators must also handle existing resources in the destination objects. In container operations, such as STD vector push back or STD sort, objects may be moved to new locations. Without move assignment operator, this could fall back to a more expensive copy assignment. Move assignment ensures that reassignment remains efficient, especially when the container needs to reorder or compact its elements. 

## Encapsulation 

````C++
class Employee{
private:
    string company; 
    string surname;
    string givenName; 

public:
    const string& getCompany() const; 
    void setCompany(const string& newCompany); 

    const string& getSurname() const; 
    void setSurname(const string& newSurname); 

    const string& getGivenName() const;
    void setGivenName(const string& newGivenName);
````

Encapsulation, which has been introduced as the practice of hiding internal state and requiring all access through well-defined interfaces, is reinforced in C++ using accessors and mutators. These public methods allow controlled access to private member variables. By defining exactly how data is retrieved and modified, we can enforce class invariants, log access or updates, and shield internal implementation details from the outside world. Accessors or getters return internal values, while mutators or setters update them.  Used together, they allow developers to maintain robust, encapsulated designs without exposing private state. 

## Accesor Functions


You may have noticed that the accessor functions return a `const`reference to the underlying data, and that they have the keyword `const`after them. A member fucntion maked as `const`, promises not to change the state of the object. Can be called on const instances, and is an important part of writing safe, reliable classes. Without being makred as `const`, the compiler would prevent accessor functions from being called on const objects.

Without the const qualifier on the return value, users could modify the underlying data through the reference acquired from the accesor function. Returning a const reference guarantees that the reference is read only. In c++, every non static member function of a class has access to a pointer called this, wihich points to the object that the method was called on . 


# APPENDIX
## Shallow copy
In C++, **shallow** referes to a method of copying an object that dopies only its immediate member values, without duplicating any dynamically allocated memory it points to 

## noexcept
In C++, **`noexcept`** is a keyword that specifies whether a function will throw exceptions or not. It serves as an optimization hint for the compiler and a guarantee for developers. 

you can use `noexcept` either as a **specifier** (to declare a function) or as an **operator**(to check a function)

### The `noexcept` Specifier(Declaration)
When you append `noexcept` to a function signature, you promise that the function will never allow an exception to escape
````c++
void safeFunction() noexcept{
    // This function will not throw exceptions
}

void riskyFunction() noexcept(false){
    // this function CAN throw exceptions
}
````
- **What happens if it throws anyway?** If a function marked `noexcept` tries to throw an exception, the program immediately calls `std::terminate()` and crashes. It does not look for a `catch` block.

### The `noexcept` Operator
 You can use `noexcept` as a compile-time operator to check if an expression or function is marked as non-throwing. It returns a boolean
 ````c++
 bool isSafe = noexcept(safeFunction()); // Evaluates to true
 ````

### Importance of `noexcept`
- **Performance Optimization** When the compiler knows a function cannot throw, it can omit the heavy stack-unwinding code required to handle exceptions, resulting in smaller and faster binaries.
- **Vector Reallocation** Standard library containers like `std::vector` check if your class has a `noexcept` move constructor. If it does, the vector will efficiently **move** elements during resize. If it doesn't, the vector will **copy** them instead to maintain exception safety, which is much slower.

### When to use it
- **Always use it for:** Move constructors, move assignment operators, destructors, and swap functions.
- **Avoid it for:** Functions that rely on external inputs, memory allocation (`new`), or anything that could realistically fail